In [1]:
!pip install python-chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 36.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147879 sha256=e329ba3c0ad81562b55543b65e9279de6d90f898f4906ab65afae124dfe613af
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess


In [2]:
import chess
import chess.svg
import numpy as np
import random
from sklearn.neural_network import MLPRegressor
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

In [3]:
def board_to_features(board):
    """
    Converts a chess.Board object into a 1D numpy array of size 64 * 12 = 768.
    We use one-hot encoding for the 12 possible piece types (6 white, 6 black).
    """
    features = np.zeros(768, dtype=np.float32)
    piece_map = {
        (chess.PAWN, chess.WHITE): 0,
        (chess.KNIGHT, chess.WHITE): 1,
        (chess.BISHOP, chess.WHITE): 2,
        (chess.ROOK, chess.WHITE): 3,
        (chess.QUEEN, chess.WHITE): 4,
        (chess.KING, chess.WHITE): 5,
        (chess.PAWN, chess.BLACK): 6,
        (chess.KNIGHT, chess.BLACK): 7,
        (chess.BISHOP, chess.BLACK): 8,
        (chess.ROOK, chess.BLACK): 9,
        (chess.QUEEN, chess.BLACK): 10,
        (chess.KING, chess.BLACK): 11
    }

    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is not None:
            index = piece_map[(piece.piece_type, piece.color)]
            features[square * 12 + index] = 1.0
    return features

def material_heuristic(board):
    """
    Simple material evaluation: positive scores favor white, negative favor black.
    """
    if board.is_checkmate():
        return -9999 if board.turn == chess.WHITE else 9999
    if board.is_game_over():
        return 0

    values = {
        chess.PAWN: 100,
        chess.KNIGHT: 320,
        chess.BISHOP: 330,
        chess.ROOK: 500,
        chess.QUEEN: 900,
        chess.KING: 20000
    }
    score = 0
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is not None:
            val = values[piece.piece_type]
            if piece.color == chess.WHITE:
                score += val
            else:
                score -= val
    return score

def generate_dataset(num_samples=3000):
    """
    Generates random game positions and calculates their material score.
    """
    X = []
    y = []
    for _ in range(num_samples):
        board = chess.Board()
        # Make a random number of random moves to diversify positions
        for _ in range(random.randint(5, 45)):
            if board.is_game_over():
                break
            legal_moves = list(board.legal_moves)
            if not legal_moves:
                break
            board.push(random.choice(legal_moves))

        X.append(board_to_features(board))
        y.append(material_heuristic(board))
    return np.array(X), np.array(y)

In [4]:
print("Generating chess dataset (this will take a few seconds)... ")
X_train, y_train = generate_dataset(4000)
print(f"Dataset ready. Features shape: {X_train.shape}, Labels shape: {y_train.shape}")

print("Training MLP Neural Network Model...")
ann_evaluator = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=150,
    random_state=42
)
ann_evaluator.fit(X_train, y_train)
print("ANN Training complete!")

# Quick test
test_board = chess.Board()
test_features = board_to_features(test_board).reshape(1, -1)
pred_score = ann_evaluator.predict(test_features)[0]
print(f"Initial board predicted evaluation score: {pred_score:.2f} (Should be close to 0)")

Generating chess dataset (this will take a few seconds)... 
Dataset ready. Features shape: (4000, 768), Labels shape: (4000,)
Training MLP Neural Network Model...
ANN Training complete!
Initial board predicted evaluation score: 35.24 (Should be close to 0)


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


In [5]:
def evaluate_board_ann(board):
    """
    Evaluates the board layout using our trained Artificial Neural Network.
    """
    if board.is_checkmate():
        return -99999 if board.turn == chess.WHITE else 99999
    if board.is_game_over():
        return 0
    features = board_to_features(board).reshape(1, -1)
    return ann_evaluator.predict(features)[0]

def minimax(board, depth, alpha, beta, maximizing_player):
    if depth == 0 or board.is_game_over():
        return evaluate_board_ann(board), None

    best_move = None
    if maximizing_player:
        max_eval = -float('inf')
        for move in board.legal_moves:
            board.push(move)
            evaluation, _ = minimax(board, depth - 1, alpha, beta, False)
            board.pop()
            if evaluation > max_eval:
                max_eval = evaluation
                best_move = move
            alpha = max(alpha, evaluation)
            if beta <= alpha:
                break
        return max_eval, best_move
    else:
        min_eval = float('inf')
        for move in board.legal_moves:
            board.push(move)
            evaluation, _ = minimax(board, depth - 1, alpha, beta, True)
            board.pop()
            if evaluation < min_eval:
                min_eval = evaluation
                best_move = move
            beta = min(beta, evaluation)
            if beta <= alpha:
                break
        return min_eval, best_move

In [6]:
class ChessGame:
    def __init__(self):
        self.board = chess.Board()
        self.history = []
        self.search_depth = 2 # Depth for the Minimax evaluation

        # Layout & Widget definitions
        self.board_output = widgets.Output()
        self.status_output = widgets.Output()
        self.move_input = widgets.Text(
            placeholder='Type a move (e.g., e2e4, g1f3)',
            description='Your Move:',
            disabled=False
        )
        self.submit_btn = widgets.Button(description="Submit Move", button_style='primary')
        self.undo_btn = widgets.Button(description="Undo Move", button_style='warning')
        self.reset_btn = widgets.Button(description="New Game", button_style='danger')

        # Click handlers
        self.submit_btn.on_click(self.on_submit_move)
        self.move_input.on_submit(self.on_submit_move)
        self.undo_btn.on_click(self.on_undo)
        self.reset_btn.on_click(self.on_reset)

        self.update_view()

    def update_view(self):
        with self.board_output:
            clear_output(wait=True)
            # Render SVG representation of the board
            svg_data = chess.svg.board(self.board, size=400)
            display(HTML(svg_data))

        with self.status_output:
            clear_output(wait=True)
            print(f"Current Turn: {'White (You)' if self.board.turn == chess.WHITE else 'Black (AI)'}")
            print(f"Move History: {', '.join([m.uci() for m in self.history])}")

            if self.board.is_checkmate():
                print("GAME OVER: Checkmate!")
            elif self.board.is_stalemate():
                print("GAME OVER: Stalemate!")
            elif self.board.is_game_over():
                print("GAME OVER: Draw or other finish condition.")
            elif self.board.is_check():
                print("Warning: Check!")

    def make_ai_move(self):
        if self.board.is_game_over():
            return

        # AI plays as Black (Minimizer)
        _, ai_move = minimax(self.board, self.search_depth, -float('inf'), float('inf'), False)
        if ai_move:
            self.board.push(ai_move)
            self.history.append(ai_move)
            self.update_view()

    def on_submit_move(self, b):
        user_move_str = self.move_input.value.strip()
        self.move_input.value = '' # Reset field

        try:
            move = chess.Move.from_uci(user_move_str)
            # Handle pawn promotion defaults to Queen if move matches length 4
            if len(user_move_str) == 4:
                from_sq = chess.parse_square(user_move_str[:2])
                piece = self.board.piece_at(from_sq)
                if piece and piece.piece_type == chess.PAWN and user_move_str[3] in ['8', '1']:
                    move = chess.Move.from_uci(user_move_str + 'q') # auto promote to queen

            if move in self.board.legal_moves:
                self.board.push(move)
                self.history.append(move)
                self.update_view()

                # Let the AI make its decision asynchronously
                if not self.board.is_game_over():
                    self.make_ai_move()
            else:
                with self.status_output:
                    print(f"Illegal Move: {user_move_str}. Please try a valid move.")
        except Exception as e:
            with self.status_output:
                print("Invalid format! Use coordinate notation (e.g., e2e4).")

    def on_undo(self, b):
        # Undo user move and AI move
        if len(self.history) >= 2:
            self.board.pop()
            self.board.pop()
            self.history.pop()
            self.history.pop()
        elif len(self.history) == 1:
            self.board.pop()
            self.history.pop()
        self.update_view()

    def on_reset(self, b):
        self.board = chess.Board()
        self.history = []
        self.update_view()

    def show_interface(self):
        controls = widgets.VBox([
            self.move_input,
            widgets.HBox([self.submit_btn, self.undo_btn, self.reset_btn])
        ])
        display(widgets.HBox([self.board_output, widgets.VBox([self.status_output, controls])]))

In [7]:
game = ChessGame()
game.show_interface()